In [ ]:
import numpy as np
import pandas as pd
import joblib
from datetime import date, datetime
from pathlib import Path
from astro_calculator import compute_all_astro_features

In [3]:
pipeline = joblib.load('best_pipeline.pkl')
feature_cols = joblib.load('feature_columns.pkl')

FileNotFoundError: [Errno 2] No such file or directory: 'best_pipeline.pkl'

In [ ]:


# ============================================================
# 1. КОНФИГУРАЦИЯ
# ============================================================

FEATURE_COLS = [
    'moon_phase_sin',
    'moon_phase_cos',
    'mercury_retro',
    'angle_merc_ven',
    'is_weekend',
]

MODEL_PATH = 'best_multi_output_model.pkl'
SCALER_PATH = 'scaler.pkl'

# ============================================================
# 2. ЗАГРУЗКА МОДЕЛИ
# ============================================================

def load_model(model_path: str = MODEL_PATH):
    """Загружает обученную модель и скалер."""
    if not Path(model_path).exists():
        raise FileNotFoundError(
            f"Модель не найдена: {model_path}\n"
            "Сначала запустите multi_output_ml.py для обучения и сохранения модели."
        )
    
    model = joblib.load(model_path)
    print(f"✅ Модель загружена: {model_path}")
    return model

# ============================================================
# 3. ФУНКЦИЯ ПРЕДСКАЗАНИЯ
# ============================================================

def predict_consumption(input_date: date, model) -> dict:
    """
    Предсказывает потребление кофе и пива для заданной даты.
    
    Parameters
    ----------
    input_date : date
        Дата для предсказания.
    model : sklearn.pipeline.Pipeline
        Обученная модель (включает скалер).
    
    Returns
    -------
    dict
        Словарь с предсказаниями: {'coffee_rub': float, 'beer_rub': float}
    """
    # Рассчитываем астрологические признаки
    astro_features = compute_all_astro_features(input_date)
    
    # Извлекаем только нужные признаки в правильном порядке
    X = np.array([[astro_features[col] for col in FEATURE_COLS]])
    
    # Делаем предсказание
    y_pred = model.predict(X)
    
    # Извлекаем значения (модель возвращает массив формы (1, 2))
    coffee_pred = float(y_pred[0, 0])
    beer_pred = float(y_pred[0, 1])
    
    # Ограничиваем отрицательные значения (нельзя потратить меньше нуля)
    coffee_pred = max(0, coffee_pred)
    beer_pred = max(0, beer_pred)
    
    return {
        'date': input_date,
        'coffee_rub': coffee_pred,
        'beer_rub': beer_pred,
        'astro_features': astro_features,
    }

def predict_multiple_dates(dates: list[date], model) -> pd.DataFrame:
    """
    Предсказывает потребление для списка дат.
    
    Returns
    -------
    pd.DataFrame
        Таблица с предсказаниями и астрологическими признаками.
    """
    results = []
    for d in dates:
        pred = predict_consumption(d, model)
        results.append(pred)
    
    df = pd.DataFrame(results)
    return df

# ============================================================
# 4. ИНТЕРПРЕТАЦИЯ РЕЗУЛЬТАТОВ
# ============================================================

def interpret_prediction(pred: dict) -> str:
    """Генерирует текстовую интерпретацию предсказания."""
    date = pred['date']
    coffee = pred['coffee_rub']
    beer = pred['beer_rub']
    astro = pred['astro_features']
    
    # Определяем фазу луны
    moon_phase = astro['moon_phase']
    if moon_phase < 0.125 or moon_phase >= 0.875:
        moon_text = "🌑 Новолуние"
    elif moon_phase < 0.375:
        moon_text = "🌓 Первая четверть"
    elif moon_phase < 0.625:
        moon_text = "🌕 Полнолуние"
    else:
        moon_text = "🌗 Последняя четверть"
    
    # Ретроградный Меркурий
    mercury_text = "☿️ Ретроградный" if astro['mercury_retro'] else "☿️ Директный"
    
    # Выходной
    weekend_text = "🎉 Выходной" if astro['is_weekend'] else "💼 Будний день"
    
    # Формируем отчёт
    report = f"""
╔══════════════════════════════════════════════════════════╗
║  КОСМО-БАР: ПРОГНОЗ НА {date.strftime('%d.%m.%Y')}              ║
╠══════════════════════════════════════════════════════════╣
║  Астрологическая обстановка:                             ║
║  • {moon_text:<50} ║
║  • {mercury_text:<50} ║
║  • {weekend_text:<50} ║
║  • Угол Меркурий-Венера: {astro['angle_merc_ven']:.1f}°{' '*28} ║
╠══════════════════════════════════════════════════════════╣
║  Прогноз потребления:                                    ║
║  ☕  Кофе:  {coffee:.2f} руб.{' '*35} ║
║  🍺  Пиво:  {beer:.2f} руб.{' '*35} ║
╠══════════════════════════════════════════════════════════╣
║  Рекомендация:                                           ║
"""
    
    # Генерируем рекомендацию
    if coffee > 3.0 and beer > 5.0:
        recommendation = "⚠️  Высокий риск гиперфиксации. Рекомендуется Земфира в наушниках."
    elif coffee > 3.0:
        recommendation = "☕  День продуктивной апатии. Запаситесь Nescafe Crema."
    elif beer > 5.0:
        recommendation = "🍺  Вечер обещает быть интересным. Контролируйте метрику пространства-времени."
    elif astro['mercury_retro']:
        recommendation = "☿️  Ретроградный Меркурий. Избегайте важных решений и новых коктейлей."
    else:
        recommendation = "✅  Метрика стабильна. Можно продолжать анализ экспериментальных данных."
    
    report += f"║  {recommendation:<58} ║\n"
    report += "╚══════════════════════════════════════════════════════════╝"
    
    return report

# ============================================================
# 5. ГЛАВНАЯ ФУНКЦИЯ
# ============================================================

def main():
    print("="*70)
    print("КОСМО-БАР: СИСТЕМА ПРЕДСКАЗАНИЯ ПОТРЕБЛЕНИЯ v1.0")
    print("="*70)
    
    # Загрузка модели
    model = load_model()
    
    # Пример 1: Предсказание на сегодня
    today = date.today()
    print(f"\n🔮 Предсказание на сегодня ({today.strftime('%d.%m.%Y')}):")
    pred_today = predict_consumption(today, model)
    print(interpret_prediction(pred_today))
    
    # Пример 2: Предсказание на неделю вперёд
    print("\n" + "="*70)
    print("ПРОГНОЗ НА НЕДЕЛЮ")
    print("="*70)
    
    next_week = [today + pd.Timedelta(days=i) for i in range(7)]
    df_week = predict_multiple_dates(next_week, model)
    
    print("\n📊 Таблица предсказаний:")
    print(df_week[['date', 'coffee_rub', 'beer_rub']].to_string(index=False))
    
    # Сохранение результатов
    output_path = 'predictions_next_week.csv'
    df_week.to_csv(output_path, index=False)
    print(f"\n✅ Результаты сохранены: {output_path}")

if __name__ == '__main__':
    main()

ModuleNotFoundError: No module named 'astro_calculator'